# Lab: Neural Networks Fundamentals

This notebook implements forward propagation for a single neuron and a small MLP in NumPy, recreates the same network in PyTorch, and compares hidden-layer activation functions.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
torch.manual_seed(42)

## Task 1 — A Single Neuron in NumPy

In [ ]:
# Load dataset
data = load_breast_cancer()
X = data.data
y = data.target

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

# Train/test split and scaling
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train size: {X_train_scaled.shape[0]}, Test size: {X_test_scaled.shape[0]}")

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def forward(x, w, b):
    """Single-neuron forward pass: sigmoid(w·x + b)."""
    z = np.dot(x, w) + b
    return sigmoid(z)


# Initialize weights from a small random distribution
w = np.random.randn(30) * 0.01
b = 0.0

# Run on the first 5 test rows
X_sample = X_test_scaled[:5]
probs = np.array([forward(x, w, b) for x in X_sample])

print("Predicted probabilities for the first 5 test rows:")
for i, p in enumerate(probs):
    print(f"  Sample {i}: {p:.6f}")

### Task 1 — Interpretation

This single-neuron forward pass is **logistic regression** — a single artificial neuron with a sigmoid activation for **binary classification**. Each output is a probability in $(0, 1)$ for the positive class (malignant vs. benign). Without training, the probabilities are arbitrary, but the structure matches what we use for binary decisions after fitting weights.

## Task 2 — A Two-Layer MLP in NumPy

In [ ]:
class NumpyMLP:
    def __init__(self, input_size=30, hidden_size=8, output_size=1):
        # He initialization: W ~ N(0, sqrt(2 / fan_in))
        self.W1 = np.random.randn(input_size, hidden_size) * np.sqrt(2.0 / input_size)
        self.b1 = np.zeros(hidden_size)
        self.W2 = np.random.randn(hidden_size, output_size) * np.sqrt(2.0 / hidden_size)
        self.b2 = np.zeros(output_size)

    @staticmethod
    def relu(z):
        return np.maximum(0, z)

    @staticmethod
    def sigmoid(z):
        return 1.0 / (1.0 + np.exp(-z))

    def forward(self, X):
        """Vectorized forward pass. X shape: (N, input_size) -> (N, 1)."""
        z1 = X @ self.W1 + self.b1          # (N, 8)
        a1 = self.relu(z1)                  # (N, 8)
        z2 = a1 @ self.W2 + self.b2         # (N, 1)
        return self.sigmoid(z2)             # (N, 1)


numpy_mlp = NumpyMLP(input_size=30, hidden_size=8, output_size=1)
numpy_preds = numpy_mlp.forward(X_test_scaled)

print(f"Output shape: {numpy_preds.shape}")
print("First 5 predictions:")
print(numpy_preds[:5].flatten())

### Task 2 — Interpretation

The output shape **(N, 1)** gives one predicted probability per sample — exactly what a binary classifier needs. Each row is a score in $(0, 1)$ that can be thresholded (e.g., at 0.5) into class labels. Even without training, the architecture is set up for batch binary prediction: $N$ samples in, $N$ probabilities out.

## Task 3 — The Same Network in PyTorch

In [ ]:
class TorchMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(30, 8)
        self.fc2 = nn.Linear(8, 1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.sigmoid(self.fc2(x))
        return x


torch_mlp = TorchMLP()

# Copy NumPy weights into PyTorch (nn.Linear stores weight as (out_features, in_features))
torch_mlp.fc1.weight.data = torch.from_numpy(numpy_mlp.W1.T).float()
torch_mlp.fc1.bias.data = torch.from_numpy(numpy_mlp.b1).float()
torch_mlp.fc2.weight.data = torch.from_numpy(numpy_mlp.W2.T).float()
torch_mlp.fc2.bias.data = torch.from_numpy(numpy_mlp.b2).float()

X_test_tensor = torch.from_numpy(X_test_scaled).float()

with torch.no_grad():
    torch_preds = torch_mlp(X_test_tensor).numpy()

max_diff = np.max(np.abs(torch_preds - numpy_preds))
print(f"Maximum absolute difference: {max_diff:.10f}")
print(f"Outputs match to 6+ decimal places: {max_diff < 1e-6}")
print("\nFirst 5 NumPy predictions:", numpy_preds[:5].flatten())
print("First 5 PyTorch predictions:", torch_preds[:5].flatten())

### Task 3 — Interpretation

When weights are copied correctly — remembering that `nn.Linear` stores weights as **(out_features, in_features)**, so NumPy's `(in, out)` matrix must be transposed — the PyTorch and NumPy forward passes produce **numerically identical** outputs. This confirms that both implementations encode the same computation: linear layers, ReLU, and sigmoid in the same order.

## Task 4 — Activation Function Experiment

In [ ]:
class FlexibleMLP(nn.Module):
    """Same 30 -> 8 -> 1 architecture with a configurable hidden activation."""

    def __init__(self, activation="relu"):
        super().__init__()
        self.fc1 = nn.Linear(30, 8)
        self.fc2 = nn.Linear(8, 1)
        activations = {
            "sigmoid": nn.Sigmoid(),
            "tanh": nn.Tanh(),
            "relu": nn.ReLU(),
        }
        self.hidden_activation = activations[activation]
        self.activation_name = activation

    def forward(self, x, return_hidden=False):
        z1 = self.fc1(x)
        a1 = self.hidden_activation(z1)
        out = torch.sigmoid(self.fc2(a1))
        if return_hidden:
            return out, z1, a1
        return out


def he_init_model(model, input_size=30, hidden_size=8):
    """Apply He initialization to match the NumPy MLP setup."""
    with torch.no_grad():
        model.fc1.weight.normal_(0, np.sqrt(2.0 / input_size))
        model.fc1.bias.zero_()
        model.fc2.weight.normal_(0, np.sqrt(2.0 / hidden_size))
        model.fc2.bias.zero_()


activations = ["sigmoid", "tanh", "relu"]
hidden_pre = {}
hidden_post = {}

for act in activations:
    torch.manual_seed(42)
    model = FlexibleMLP(activation=act)
    he_init_model(model)

    with torch.no_grad():
        _, z1, a1 = model(X_test_tensor, return_hidden=True)

    hidden_pre[act] = z1.numpy().flatten()
    hidden_post[act] = a1.numpy().flatten()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, act in zip(axes, activations):
    ax.hist(hidden_pre[act], bins=30, edgecolor="black", alpha=0.7)
    ax.set_title(f"{act.capitalize()} — Pre-activations (z1)")
    ax.set_xlabel("z1 value")
    ax.set_ylabel("Count")
plt.suptitle("Hidden-layer pre-activations before activation function", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, act in zip(axes, activations):
    ax.hist(hidden_post[act], bins=30, edgecolor="black", alpha=0.7)
    ax.set_title(f"{act.capitalize()} — Post-activations (a1)")
    ax.set_xlabel("a1 value")
    ax.set_ylabel("Count")
plt.suptitle("Hidden-layer outputs after activation function", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
sigmoid_post = hidden_post["sigmoid"]
tanh_post = hidden_post["tanh"]
relu_post = hidden_post["relu"]

sigmoid_saturated = np.mean((sigmoid_post < 0.05) | (sigmoid_post > 0.95))
tanh_saturated = np.mean((tanh_post < -0.95) | (tanh_post > 0.95))
relu_inactive = np.mean(relu_post == 0.0)

print("Saturation / inactivity fractions (hidden layer post-activations):")
print(f"  Sigmoid saturated (< 0.05 or > 0.95): {sigmoid_saturated:.4f} ({sigmoid_saturated * 100:.2f}%)")
print(f"  Tanh saturated (< -0.95 or > 0.95):      {tanh_saturated:.4f} ({tanh_saturated * 100:.2f}%)")
print(f"  ReLU inactive (exactly 0):                {relu_inactive:.4f} ({relu_inactive * 100:.2f}%)")

### Task 4 — Interpretation

**Sigmoid and Tanh saturation:** A large fraction of hidden units fall in the saturated region — Sigmoid outputs pile up near 0 or 1, and Tanh outputs near ±1. In these regions the activation is nearly flat, so gradients are very small (**vanishing gradients**), which slows learning in deep networks.

**ReLU inactivity:** ReLU sets negative pre-activations to exactly 0, so a noticeable fraction of units are inactive on this forward pass. Inactive units contribute nothing to the output but can "wake up" again when weights update — unlike saturated sigmoid/tanh units that stay stuck with tiny gradients.

**Why ReLU is often the default:** ReLU is cheap to compute, keeps positive activations linear (gradient = 1), and avoids the strong saturation seen with Sigmoid/Tanh in hidden layers. That makes optimization more stable and is why ReLU (or variants like Leaky ReLU) is the usual choice for hidden layers, while Sigmoid is still common on the **output** layer for binary classification.

## Final Checklist

- [x] Single-neuron forward pass implemented and run.
- [x] NumPy MLP implemented with He initialization.
- [x] PyTorch MLP created with manually copied NumPy weights.
- [x] NumPy and PyTorch outputs match numerically.
- [x] Activation experiment completed with histograms and interpretation.
- [x] Kernel Restart & Run All should run without errors.